# JobLens: Resume–Job Matching Pipeline

In [ ]:
# Colab에서 처음 한 번만 실행하세요.
# !pip install -q sentence-transformers datasets kagglehub pandas scikit-learn


In [ ]:
from __future__ import annotations

import gc
import random
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from datasets import load_dataset
from kagglehub import KaggleDatasetAdapter
import kagglehub
from sentence_transformers import (
    InputExample, SentenceTransformer, evaluation, losses
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from torch.utils.data import DataLoader

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [ ]:
@dataclass(frozen=True)
class PipelineConfig:
    base_model_name: str = 'all-MiniLM-L6-v2'
    drive_model_path: str = '/content/drive/MyDrive/joblens-finetuned-sbert'
    local_model_path: str = 'joblens-finetuned-sbert'
    max_jobs: int = 50
    max_resumes: int = 100
    top_k: int = 3
    batch_size: int = 16
    epochs: int = 3
    device: str = 'cpu'


CONFIG = PipelineConfig()

# Resume Atlas category -> keyword expected in a LinkedIn posting title.
JOB_CATEGORY_KEYWORDS = {
    'Accountant': 'accountant', 'Advocate': 'advocate', 'Agriculture': 'agricultur',
    'Apparel': 'apparel', 'Architecture': 'architect', 'Arts': 'arts',
    'Automobile': 'automobil', 'Aviation': 'aviation', 'Banking': 'banking',
    'Blockchain': 'blockchain', 'BPO': 'bpo',
    'Building and Construction': 'construction', 'Business Analyst': 'business analyst',
    'Civil Engineer': 'civil engineer', 'Consultant': 'consultant',
    'Data Science': 'data scien', 'Database': 'database', 'Designing': 'design',
    'DevOps': 'devops', 'Digital Media': 'digital media',
    'DotNet Developer': 'dotnet', 'Education': 'education',
    'Electrical Engineering': 'electrical', 'ETL Developer': 'etl',
    'Finance': 'financ', 'Food and Beverages': 'food',
    'Health and Fitness': 'health', 'Human Resources': 'human resource',
    'Information Technology': 'information technolog', 'Java Developer': 'java',
    'Management': 'management', 'Mechanical Engineer': 'mechanical',
    'Network Security Engineer': 'network security',
    'Operations Manager': 'operations', 'PMO': 'pmo',
    'Public Relations': 'public relation', 'Python Developer': 'python',
    'React Developer': 'react', 'Sales': 'sales', 'SAP Developer': 'sap',
    'SQL Developer': 'sql', 'Testing': 'test', 'Web Designing': 'web design',
}


## 데이터 준비 함수

데이터 로딩, 정제, ground-truth 구성만 담당합니다.

In [ ]:
def load_resume_dataset() -> pd.DataFrame:
    """Hugging Face Resume Atlas를 표준 resume DataFrame으로 변환한다."""
    raw_resumes = load_dataset('ahmedheakl/resume-atlas')['train'].to_pandas()
    return pd.DataFrame({
        'id': [f'R{i}' for i in range(len(raw_resumes))],
        'category': raw_resumes['Category'].fillna('').str.strip(),
        'resume_text': raw_resumes['Text'].fillna('').astype(str),
    })


def load_job_postings() -> pd.DataFrame:
    """Kaggle LinkedIn postings를 표준 job DataFrame으로 변환한다."""
    raw_jobs = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS, 'arshkon/linkedin-job-postings', 'postings.csv'
    )
    return pd.DataFrame({
        'id': [f'J{i}' for i in range(len(raw_jobs))],
        'title': raw_jobs['title'].fillna('').astype(str),
        'job_description': raw_jobs['description'].fillna('').astype(str),
    })


def infer_job_category(job_title: str) -> str | None:
    """제목 키워드로 평가용 resume category를 추정한다."""
    normalized_title = str(job_title).lower()
    return next((category for category, keyword in JOB_CATEGORY_KEYWORDS.items()
                 if keyword in normalized_title), None)


def prepare_evaluation_data(
    resumes: pd.DataFrame, jobs: pd.DataFrame, config: PipelineConfig
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """category별 정답 resume를 보장하는 작고 재현 가능한 평가 세트를 만든다."""
    tagged_jobs = jobs.copy()
    tagged_jobs['matched_category'] = tagged_jobs['title'].map(infer_job_category)
    tagged_jobs = tagged_jobs.dropna(subset=['matched_category'])

    category_to_resume_id = (
        resumes.groupby('category')['id'].sample(n=1, random_state=RANDOM_SEED).to_dict()
    )
    # Series.to_dict() above is id -> id; construct an explicit category mapping instead.
    category_to_resume_id = (
        resumes.groupby('category', group_keys=False).apply(
            lambda group: group.sample(n=1, random_state=RANDOM_SEED)
        ).set_index('category')['id'].to_dict()
    )
    tagged_jobs['correct_resume_id'] = tagged_jobs['matched_category'].map(category_to_resume_id)
    tagged_jobs = (tagged_jobs.dropna(subset=['correct_resume_id'])
                   .sample(n=min(config.max_jobs, len(tagged_jobs)), random_state=RANDOM_SEED)
                   .reset_index(drop=True))

    required_ids = set(tagged_jobs['correct_resume_id'])
    required_resumes = resumes[resumes['id'].isin(required_ids)]
    optional_resumes = resumes[~resumes['id'].isin(required_ids)]
    remaining_count = max(0, config.max_resumes - len(required_resumes))
    sampled_optional = optional_resumes.sample(
        n=min(remaining_count, len(optional_resumes)), random_state=RANDOM_SEED
    )
    evaluation_resumes = pd.concat([required_resumes, sampled_optional], ignore_index=True)
    return evaluation_resumes, tagged_jobs


## 매칭·평가 함수

모델 임베딩, 유사도 행렬, Precision@K/NDCG@K 계산을 담당합니다.

In [ ]:
def build_tfidf_similarity_matrix(resumes: pd.DataFrame, jobs: pd.DataFrame) -> np.ndarray:
    """TF-IDF cosine similarity matrix: (job count, resume count)."""
    corpus = resumes['resume_text'].tolist() + jobs['job_description'].tolist()
    vectors = TfidfVectorizer(stop_words='english').fit_transform(corpus)
    return cosine_similarity(vectors[len(resumes):], vectors[:len(resumes)])


def build_embedding_similarity_matrix(
    model: SentenceTransformer, resumes: pd.DataFrame, jobs: pd.DataFrame, batch_size: int
) -> np.ndarray:
    """SentenceTransformer로 job-resume cosine similarity matrix를 만든다."""
    resume_embeddings = model.encode(
        resumes['resume_text'].tolist(), batch_size=batch_size, show_progress_bar=False
    )
    job_embeddings = model.encode(
        jobs['job_description'].tolist(), batch_size=batch_size, show_progress_bar=False
    )
    return cosine_similarity(job_embeddings, resume_embeddings)


def calculate_ranking_metrics(
    similarity_matrix: np.ndarray, jobs: pd.DataFrame, resumes: pd.DataFrame, top_k: int
) -> Dict[str, object]:
    """single relevant resume 기준 Precision@K, NDCG@K와 상세 순위를 반환한다."""
    resume_ids = resumes['id'].tolist()
    hits, ndcg_scores, details = 0, [], []
    for row_index, job in jobs.iterrows():
        ranked_indices = np.argsort(similarity_matrix[row_index])[::-1]
        ranked_ids = [resume_ids[index] for index in ranked_indices]
        correct_id = job['correct_resume_id']
        rank = ranked_ids.index(correct_id) + 1 if correct_id in ranked_ids else None
        hit = rank is not None and rank <= top_k
        hits += int(hit)
        ndcg_scores.append(1 / np.log2(rank + 1) if hit else 0.0)
        details.append({
            'job_title': job['title'], 'correct_resume': correct_id,
            'top1_predicted': ranked_ids[0],
            'top1_score': round(float(similarity_matrix[row_index, ranked_indices[0]]), 3),
            f'hit_at_{top_k}': hit,
        })
    return {
        f'Precision@{top_k}': round(hits / len(jobs), 3),
        f'NDCG@{top_k}': round(float(np.mean(ndcg_scores)), 3),
        'details': pd.DataFrame(details),
    }


def evaluate_similarity_model(
    model_name: str, similarity_matrix: np.ndarray, jobs: pd.DataFrame, resumes: pd.DataFrame, top_k: int
) -> Dict[str, object]:
    metrics = calculate_ranking_metrics(similarity_matrix, jobs, resumes, top_k)
    return {'Model': model_name, **metrics}


## Fine-tuned 모델 관리 함수

Drive 모델이 있으면 즉시 로드합니다. Drive에 없으면 학습하고 로컬 및 Drive에 저장합니다.

In [ ]:
def mount_google_drive() -> bool:
    """Colab일 때만 Google Drive를 mount하고 성공 여부를 반환한다."""
    try:
        from google.colab import drive
    except ImportError:
        print('Google Colab 환경이 아니므로 Drive를 건너뜁니다.')
        return False
    drive.mount('/content/drive', force_remount=False)
    return True


def model_files_exist(model_path: str | Path) -> bool:
    """SentenceTransformer 저장 폴더인지 최소 파일 기준으로 확인한다."""
    path = Path(model_path)
    return path.is_dir() and (path / 'config_sentence_transformers.json').exists()


def create_training_examples(resumes: pd.DataFrame, jobs: pd.DataFrame) -> List[InputExample]:
    """직무-동일 category resume는 positive, 다른 category는 negative pair로 만든다."""
    resumes_by_category = resumes.groupby('category')['resume_text'].apply(list).to_dict()
    examples: List[InputExample] = []
    rng = random.Random(RANDOM_SEED)
    for _, job in jobs.iterrows():
        category = job['matched_category']
        positives = resumes_by_category.get(category, [])
        if not positives:
            continue
        job_text = job['job_description'][:512]
        for resume_text in rng.sample(positives, k=min(2, len(positives))):
            examples.append(InputExample(texts=[job_text, resume_text[:512]], label=1.0))
        other_categories = [key for key in resumes_by_category if key != category]
        for negative_category in rng.sample(other_categories, k=min(4, len(other_categories))):
            negative_resume = rng.choice(resumes_by_category[negative_category])
            examples.append(InputExample(texts=[job_text, negative_resume[:512]], label=0.0))
    rng.shuffle(examples)
    if len(examples) < 2:
        raise ValueError('Fine-tuning에 필요한 학습 쌍이 충분하지 않습니다.')
    return examples


def train_finetuned_model(
    resumes: pd.DataFrame, jobs: pd.DataFrame, config: PipelineConfig
) -> SentenceTransformer:
    """base model을 fine-tune하고 local_model_path에 best model을 저장한다."""
    examples = create_training_examples(resumes, jobs)
    split_index = max(1, int(len(examples) * 0.9))
    train_examples, validation_examples = examples[:split_index], examples[split_index:]
    model = SentenceTransformer(config.base_model_name, device=config.device)
    train_loader = DataLoader(train_examples, shuffle=True, batch_size=config.batch_size)
    evaluator = None
    if validation_examples:
        evaluator = evaluation.EmbeddingSimilarityEvaluator(
            [item.texts[0] for item in validation_examples],
            [item.texts[1] for item in validation_examples],
            [item.label for item in validation_examples], name='validation'
        )
    model.fit(
        train_objectives=[(train_loader, losses.CosineSimilarityLoss(model))],
        evaluator=evaluator, epochs=config.epochs,
        warmup_steps=max(1, int(len(train_loader) * 0.1)),
        evaluation_steps=max(1, int(len(train_loader) * 0.5)),
        output_path=config.local_model_path, save_best_model=True, show_progress_bar=True,
    )
    return SentenceTransformer(config.local_model_path, device=config.device)


def load_or_train_finetuned_model(
    resumes: pd.DataFrame, jobs: pd.DataFrame, config: PipelineConfig
) -> Tuple[SentenceTransformer, str]:
    """Drive -> local cache -> fine-tune 순으로 모델을 확보한다."""
    drive_is_available = mount_google_drive()
    if drive_is_available and model_files_exist(config.drive_model_path):
        print(f'✅ Google Drive fine-tuned 모델 로드: {config.drive_model_path}')
        return SentenceTransformer(config.drive_model_path, device=config.device), 'Google Drive'
    if model_files_exist(config.local_model_path):
        print(f'✅ 로컬 fine-tuned 모델 로드: {config.local_model_path}')
        return SentenceTransformer(config.local_model_path, device=config.device), 'local cache'

    print('ℹ️ 저장된 fine-tuned 모델이 없어 학습을 시작합니다.')
    model = train_finetuned_model(resumes, jobs, config)
    if drive_is_available:
        destination = Path(config.drive_model_path)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(config.local_model_path, destination, dirs_exist_ok=True)
        print(f'✅ 학습 모델을 Google Drive에 저장: {destination}')
    return model, 'trained this run'


## Run All 파이프라인

아래 마지막 셀 하나가 모든 단계를 의존성 순서대로 호출합니다. 필요하면 `PipelineConfig` 값만 조정하세요.

In [ ]:
def evaluate_pretrained_models(
    resumes: pd.DataFrame, jobs: pd.DataFrame, config: PipelineConfig
) -> List[Dict[str, object]]:
    """TF-IDF 및 사전학습 SBERT 모델의 결과를 수집한다."""
    results = [evaluate_similarity_model(
        'TF-IDF (Baseline)', build_tfidf_similarity_matrix(resumes, jobs),
        jobs, resumes, config.top_k
    )]
    model_ids = ['all-MiniLM-L6-v2', 'all-mpnet-base-v2', 'BAAI/bge-base-en-v1.5']
    for model_id in model_ids:
        print(f'평가 중: {model_id}')
        model = SentenceTransformer(model_id, device=config.device)
        similarity = build_embedding_similarity_matrix(model, resumes, jobs, config.batch_size)
        results.append(evaluate_similarity_model(model_id, similarity, jobs, resumes, config.top_k))
        del model, similarity
        gc.collect()
    return results


def run_job_matching_pipeline(config: PipelineConfig = CONFIG) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run All entry point: load -> prepare -> Drive/load/train -> evaluate -> save."""
    print('1/5 Resume 및 job posting 데이터 로드')
    all_resumes = load_resume_dataset()
    all_jobs = load_job_postings()

    print('2/5 평가 데이터 준비')
    resumes, jobs = prepare_evaluation_data(all_resumes, all_jobs, config)
    print(f'   resumes={len(resumes)}, jobs={len(jobs)}')

    print('3/5 fine-tuned 모델 확보 (Google Drive 우선)')
    finetuned_model, model_source = load_or_train_finetuned_model(resumes, jobs, config)

    print('4/5 baseline 및 사전학습 모델 평가')
    results = evaluate_pretrained_models(resumes, jobs, config)

    print('5/5 fine-tuned 모델 평가 및 결과 저장')
    finetuned_similarity = build_embedding_similarity_matrix(
        finetuned_model, resumes, jobs, config.batch_size
    )
    results.append(evaluate_similarity_model(
        f'SBERT Fine-tuned (JobLens; {model_source})', finetuned_similarity,
        jobs, resumes, config.top_k
    ))

    metric_columns = ['Model', f'Precision@{config.top_k}', f'NDCG@{config.top_k}']
    comparison = pd.DataFrame(results)[metric_columns]
    output_file = f'comparison_models_pool{len(resumes)}.csv'
    comparison.to_csv(output_file, index=False)
    print('\n' + comparison.to_string(index=False))
    print(f'\n✅ 비교 결과 저장: {output_file}')
    return comparison, pd.DataFrame(results)[['Model', 'details']]


# Run All 시 이 셀이 마지막으로 실행되며, 함수들을 순서대로 호출합니다.
comparison_df, ranking_details_df = run_job_matching_pipeline()
